In [1]:
import xarray as xr
import numpy as np
import netCDF4
import pandas as pd
from scipy import signal

In [2]:
var = "thetao"
arr = []
for ens_index in range(1,17):
        filename = "/work/uo1075/u241321/data/u241321/data_cdo/thetao_assi/"+var+"_Omon_MPI-ESM-LR_asSEIKERAf_r"+str(ens_index)+"i8p4_196001-202010_ym_g1d0_NA.nc"
        arr.append(xr.open_dataset(filename,decode_times=False).assign_coords(ensemble=ens_index))

In [3]:
data = xr.concat(arr, dim="member")

<xarray.Dataset>
Dimensions:    (time: 61, member: 16, bnds: 2, lon: 150, lat: 90, depth: 40)
Coordinates:
  * time       (time) float64 2.152e+04 3.028e+04 ... 5.387e+05 5.464e+05
  * lon        (lon) float64 -79.5 -78.5 -77.5 -76.5 ... 66.5 67.5 68.5 69.5
  * lat        (lat) float64 0.5 1.5 2.5 3.5 4.5 ... 85.5 86.5 87.5 88.5 89.5
  * depth      (depth) float64 6.0 17.0 27.0 37.0 ... 4.67e+03 5.17e+03 5.72e+03
    ensemble   (member) int64 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
Dimensions without coordinates: member, bnds
Data variables:
    time_bnds  (member, time, bnds) float64 2.152e+04 2.556e+04 ... 5.493e+05
    thetao     (member, time, depth, lat, lon) float32 nan nan nan ... nan nan
Attributes:
    CDI:          Climate Data Interface version 2.0.6 (https://mpimet.mpg.de...
    Conventions:  CF-1.4
    MPIOM:        $Revision: cosmos/tags/mpiesm-1.2.01p5 9876 mpiom/tags/mpio...
    history:      Sat May 06 08:54:02 2023: cdo -sellonlatbox,-80,70,0,90 the...
    CDO:          Climate Data Operators version 2.0.6 (https://mpimet.mpg.de...

In [4]:
annual_pre = data ['thetao']

In [5]:
annual_prediction = annual_pre.transpose("time", "member", "depth", "lat", "lon") # 

In [6]:
dat = np.mean(annual_prediction[10:60,:,:,:,:],axis=0)
dif = annual_prediction[5::,:,:,:,:]-dat

In [7]:
dropped = dif.stack(feature=("member","depth","lat","lon")).dropna(dim="feature")
detrend = signal.detrend(dropped ,axis=0)
feature = dropped .coords["feature"]
time = dropped .coords["time"]
detrend = xr.DataArray(detrend, dims = ["time","feature"], coords = {"time":time,"feature":feature}).unstack()

In [8]:
detrend.to_netcdf("/work/uo1075/u241321/data/temperature_1965-2020_assi_dt_5000.nc")

In [9]:
dif.to_netcdf("/work/uo1075/u241321/data/temperature_1965-2020_assi_5000.nc")